# 03 — SQL & Relational Modeling

This notebook builds the relational and SQL foundation of the SupplyGuard project.

After cleaning the raw Olist datasets, the next critical step is to understand how the processed tables should be joined safely. The dataset contains information at different levels of granularity: orders, order items, payments, reviews, customers, sellers, products, and geolocation. Joining these tables without checking their relationships can multiply rows and distort business metrics.

The goal of this notebook is to define a reliable relational structure before moving into EDA, feature engineering, machine learning, or dashboarding.

This notebook focuses on:

- loading the cleaned processed tables;
- validating primary and compound keys;
- checking the main relationships between tables;
- identifying row multiplication risks;
- defining safe join rules for order-level analysis;
- creating reusable aggregated tables for items, payments, and reviews;
- supporting the project with a MySQL SQL layer and schema documentation.

This stage turns the project from a collection of cleaned CSV files into a structured analytical data model. It ensures that future analysis starts from the correct table grain and avoids inflated metrics caused by unsafe joins.

This notebook does not perform deep EDA, create the final machine learning dataset, train models, or build dashboards. Its purpose is to make the next steps safer, cleaner, and more reliable.

Important leakage note: review-related fields and final delivery outcomes may be useful for historical diagnosis, but they remain leakage-sensitive and must not be used directly as predictive features in the final machine learning model.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 60)

In [2]:
current = Path.cwd().resolve()

for path in [current, *current.parents]:
    if (path / "data").exists():
        PROJECT_ROOT = path
        break
else:
    raise FileNotFoundError("Project root not found. Expected a 'data' folder in the project structure.")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SQL_DIR = PROJECT_ROOT / "sql"
SQL_DIR.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data directory: {PROCESSED_DIR}")
print(f"SQL directory: {SQL_DIR}")

Project root: C:\Users\johan\Desktop\supplyguard-delivery-risk
Processed data directory: C:\Users\johan\Desktop\supplyguard-delivery-risk\data\processed
SQL directory: C:\Users\johan\Desktop\supplyguard-delivery-risk\sql


## Processed Data Availability

This notebook starts from the cleaned data layer created in the previous stage.

Before loading the tables, the required processed files are checked in a compact way. This avoids repeating the full data validation from earlier notebooks while still making sure that the relational modeling step starts from the expected inputs.

In [3]:
expected_processed_files = {
    "customers": "customers_clean.csv",
    "orders": "orders_clean.csv",
    "order_items": "order_items_clean.csv",
    "order_payments": "order_payments_clean.csv",
    "order_reviews": "order_reviews_clean.csv",
    "products": "products_clean.csv",
    "sellers": "sellers_clean.csv",
    "geolocation_zip_prefix": "geolocation_zip_prefix_clean.csv",
    "category_translation": "category_translation_clean.csv",
}

file_validation = pd.DataFrame([
    {"table": table, "file": file_name, "exists": (PROCESSED_DIR / file_name).exists()}
    for table, file_name in expected_processed_files.items()
])

display(file_validation)

missing_files = file_validation.loc[~file_validation["exists"], "file"].tolist()

if missing_files:
    raise FileNotFoundError(f"Missing processed files: {missing_files}")

,table,file,exists
0,customers,customers_clean.csv,True
1,orders,orders_clean.csv,True
2,order_items,order_items_clean.csv,True
3,order_payments,order_payments_clean.csv,True
4,order_reviews,order_reviews_clean.csv,True
5,products,products_clean.csv,True
6,sellers,sellers_clean.csv,True
7,geolocation_zip_prefix,geolocation_zip_prefix_clean.csv,True
8,category_translation,category_translation_clean.csv,True


## Load Cleaned Tables

The cleaned CSV files are loaded from `data/processed/`, which is the processed layer created in the previous notebook.

Before moving into relational modeling, the notebook first confirms that the expected processed files exist and that the loaded tables match the expected cleaned outputs. This is a lightweight input check, not a repeated data quality review.

No additional cleaning is performed in this step. The goal is only to load and confirm the correct processed tables before validating keys, relationships, and join strategies.

In [4]:
tables = {table: pd.read_csv(PROCESSED_DIR / file_name, low_memory=False)
    for table, file_name in expected_processed_files.items()}

customers = tables["customers"]
orders = tables["orders"]
order_items = tables["order_items"]
order_payments = tables["order_payments"]
order_reviews = tables["order_reviews"]
products = tables["products"]
sellers = tables["sellers"]
geolocation_zip_prefix = tables["geolocation_zip_prefix"]
category_translation = tables["category_translation"]

In [5]:
table_inventory = pd.DataFrame([{"table": table, "rows": df.shape[0], "columns": df.shape[1]}
    for table, df in tables.items()
]).sort_values("table").reset_index(drop=True)

display(table_inventory)

,table,rows,columns
0,category_translation,71,2
1,customers,99441,5
2,geolocation_zip_prefix,19015,6
3,order_items,112650,7
4,order_payments,103886,5
5,order_reviews,99224,7
6,orders,99441,8
7,products,32951,10
8,sellers,3095,4


## Relational Map

Before validating joins, it is important to define the role and grain of each cleaned table.

The SupplyGuard project is built around `orders`, but not every table has one row per order. Some tables describe customers or sellers, while others describe items, payments, reviews, or geographic zip prefixes.

Understanding this structure is essential because future joins must respect the level of detail of each table. Otherwise, order-level metrics can be inflated by one-to-many relationships.

In [6]:
relational_map = pd.DataFrame([
    {"table": "orders", "role": "Core fact table", "grain": "One row per order", "main_key": "order_id", 
     "notes": "Main starting point for future order-level delivery analysis."},

    {"table": "customers", "role": "Dimension table", "grain": "One row per customer_id", "main_key": "customer_id", 
     "notes": "Connects orders to customer location attributes."},

    {"table": "sellers", "role": "Dimension table", "grain": "One row per seller_id", "main_key": "seller_id", 
     "notes": "Connects order items to seller location attributes."},

    {"table": "products", "role": "Dimension table", "grain": "One row per product_id", "main_key": "product_id", 
     "notes": "Includes product attributes and English product category."},

    {"table": "order_items", "role": "Transactional detail table", "grain": "One row per item within an order", "main_key": "order_id + order_item_id", 
     "notes": "Can multiply order rows if joined directly to orders."},

    {"table": "order_payments", "role": "Transactional detail table", "grain": "One row per payment sequence within an order", "main_key": "order_id + payment_sequential",
     "notes": "Can multiply order rows when orders have multiple payments."},

    {"table": "order_reviews", "role": "Post-delivery feedback table", "grain": "One row per review-order combination", "main_key": "review_id + order_id", 
     "notes": "Useful for post-delivery analysis, but leakage-sensitive for ML."},

    {"table": "geolocation_zip_prefix", "role": "Geographic reference table", "grain": "One row per zip code prefix", "main_key": "geolocation_zip_code_prefix", 
     "notes": "Aggregated geolocation table to avoid row multiplication."},
     
    {"table": "category_translation", "role": "Reference table", "grain": "One row per product category translation", "main_key": "product_category_name",
      "notes": "Kept as reference, although products already includes English category names."}
])

display(relational_map)

,table,role,grain,main_key,notes
0,orders,Core fact table,One row per order,order_id,Main starting point for future order-level del...
1,customers,Dimension table,One row per customer_id,customer_id,Connects orders to customer location attributes.
2,sellers,Dimension table,One row per seller_id,seller_id,Connects order items to seller location attrib...
3,products,Dimension table,One row per product_id,product_id,Includes product attributes and English produc...
4,order_items,Transactional detail table,One row per item within an order,order_id + order_item_id,Can multiply order rows if joined directly to ...
5,order_payments,Transactional detail table,One row per payment sequence within an order,order_id + payment_sequential,Can multiply order rows when orders have multi...
6,order_reviews,Post-delivery feedback table,One row per review-order combination,review_id + order_id,"Useful for post-delivery analysis, but leakage..."
7,geolocation_zip_prefix,Geographic reference table,One row per zip code prefix,geolocation_zip_code_prefix,Aggregated geolocation table to avoid row mult...
8,category_translation,Reference table,One row per product category translation,product_category_name,"Kept as reference, although products already i..."


## Key Validation

After defining the relational map, the next step is to validate the keys that will support future joins.

This check focuses only on the primary and compound keys that matter for the relational layer. The goal is to confirm whether each table can be joined safely at its expected grain.

For example, `orders` should have one row per `order_id`, while `order_items`, `order_payments`, and `order_reviews` require compound keys because they can contain multiple records related to the same order.

In [7]:
key_specs = {
    "customers": ["customer_id"], "orders": ["order_id"], "sellers": ["seller_id"], "products": ["product_id"],
    "order_items": ["order_id", "order_item_id"], "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id", "order_id"], "geolocation_zip_prefix": ["geolocation_zip_code_prefix"],
    "category_translation": ["product_category_name"]
}

key_results = []

for table, key_cols in key_specs.items():
    df = tables[table]
    missing_cols = [col for col in key_cols if col not in df.columns]

    if missing_cols:
        key_results.append({"table": table, "key": " + ".join(key_cols), "rows": len(df), "unique_keys": None, "null_key_rows": None, "duplicate_key_rows": None, "status": f"Missing columns: {missing_cols}"})
        continue

    null_key_rows = df[key_cols].isna().any(axis=1).sum()
    duplicate_key_rows = df.duplicated(subset=key_cols).sum()
    unique_keys = df[key_cols].drop_duplicates().shape[0]

    key_results.append({"table": table, "key": " + ".join(key_cols), "rows": len(df), "unique_keys": unique_keys, "null_key_rows": null_key_rows, "duplicate_key_rows": duplicate_key_rows, "status": "Pass" if null_key_rows == 0 and duplicate_key_rows == 0 else "Review"})

key_validation_summary = pd.DataFrame(key_results)

display(key_validation_summary)

,table,key,rows,unique_keys,null_key_rows,duplicate_key_rows,status
0,customers,customer_id,99441,99441,0,0,Pass
1,orders,order_id,99441,99441,0,0,Pass
2,sellers,seller_id,3095,3095,0,0,Pass
3,products,product_id,32951,32951,0,0,Pass
4,order_items,order_id + order_item_id,112650,112650,0,0,Pass
5,order_payments,order_id + payment_sequential,103886,103886,0,0,Pass
6,order_reviews,review_id + order_id,99224,99224,0,0,Pass
7,geolocation_zip_prefix,geolocation_zip_code_prefix,19015,19015,0,0,Pass
8,category_translation,product_category_name,71,71,0,0,Pass


All selected primary and compound keys passed the validation checks, confirming that the cleaned tables support the expected relational grain.

## Relationship Validation

After validating the main keys, the next step is to check whether the most important foreign-key-style relationships are covered.

This section focuses on the relationships that will matter for future joins between orders, customers, items, products, sellers, payments, reviews, and geolocation zip prefixes.

The goal is not to enforce a database schema yet, but to understand whether the cleaned data behaves consistently enough to support a SQL-ready relational layer.

In [8]:
relationship_specs = [
    {"relationship": "orders.customer_id → customers.customer_id", "left_table": "orders", "left_key": "customer_id", "right_table": "customers", "right_key": "customer_id"},
    {"relationship": "order_items.order_id → orders.order_id", "left_table": "order_items", "left_key": "order_id", "right_table": "orders", "right_key": "order_id"},
    {"relationship": "order_items.product_id → products.product_id", "left_table": "order_items", "left_key": "product_id", "right_table": "products", "right_key": "product_id"},
    {"relationship": "order_items.seller_id → sellers.seller_id", "left_table": "order_items", "left_key": "seller_id", "right_table": "sellers", "right_key": "seller_id"},
    {"relationship": "order_payments.order_id → orders.order_id", "left_table": "order_payments", "left_key": "order_id", "right_table": "orders", "right_key": "order_id"},
    {"relationship": "order_reviews.order_id → orders.order_id", "left_table": "order_reviews", "left_key": "order_id", "right_table": "orders", "right_key": "order_id"},
    {"relationship": "customers.customer_zip_code_prefix → geolocation_zip_prefix.geolocation_zip_code_prefix", "left_table": "customers", "left_key": "customer_zip_code_prefix", "right_table": "geolocation_zip_prefix", "right_key": "geolocation_zip_code_prefix"},
    {"relationship": "sellers.seller_zip_code_prefix → geolocation_zip_prefix.geolocation_zip_code_prefix", "left_table": "sellers", "left_key": "seller_zip_code_prefix", "right_table": "geolocation_zip_prefix", "right_key": "geolocation_zip_code_prefix"}
]

relationship_results = []

for spec in relationship_specs:
    left, right = tables[spec["left_table"]], tables[spec["right_table"]]
    left_key, right_key = spec["left_key"], spec["right_key"]

    left_non_null = left[left_key].dropna()
    right_keys = set(right[right_key].dropna())
    unmatched_rows = (~left_non_null.isin(right_keys)).sum()
    unmatched_pct = unmatched_rows / len(left_non_null) * 100 if len(left_non_null) else 0

    relationship_results.append({
        "relationship": spec["relationship"], "left_rows": len(left), "right_rows": len(right),
        "null_fk_rows": left[left_key].isna().sum(), "unmatched_non_null_rows": unmatched_rows,
        "unmatched_non_null_pct": round(unmatched_pct, 4), "status": "Pass" if unmatched_rows == 0 else "Review"
    })

relationship_validation_summary = pd.DataFrame(relationship_results)

display(relationship_validation_summary)

,relationship,left_rows,right_rows,null_fk_rows,unmatched_non_null_rows,unmatched_non_null_pct,status
0,orders.customer_id → customers.customer_id,99441,99441,0,0,0.0000,Pass
1,order_items.order_id → orders.order_id,112650,99441,0,0,0.0000,Pass
2,order_items.product_id → products.product_id,112650,32951,0,0,0.0000,Pass
3,order_items.seller_id → sellers.seller_id,112650,3095,0,0,0.0000,Pass
4,order_payments.order_id → orders.order_id,103886,99441,0,0,0.0000,Pass
5,order_reviews.order_id → orders.order_id,99224,99441,0,0,0.0000,Pass
6,customers.customer_zip_code_prefix → geolocati...,99441,19015,0,278,0.2796,Review
7,sellers.seller_zip_code_prefix → geolocation_z...,3095,19015,0,7,0.2262,Review


### Geolocation Coverage Review

The core business relationships passed without unmatched keys.

The only relationships marked for review are the customer and seller joins to `geolocation_zip_prefix`. Since the unmatched percentages are very small, this does not block the relational model. However, a short coverage check is useful to understand the affected zip prefixes before documenting the limitation.

In [9]:
geo_prefixes = set(geolocation_zip_prefix["geolocation_zip_code_prefix"].dropna())

unmatched_customer_geo = customers.loc[~customers["customer_zip_code_prefix"].isin(geo_prefixes), ["customer_zip_code_prefix", "customer_city", "customer_state"]]
unmatched_seller_geo = sellers.loc[~sellers["seller_zip_code_prefix"].isin(geo_prefixes), ["seller_zip_code_prefix", "seller_city", "seller_state"]]

geo_coverage_review = pd.DataFrame([
    {"entity": "customers", "unmatched_rows": len(unmatched_customer_geo), "unique_unmatched_zip_prefixes": unmatched_customer_geo["customer_zip_code_prefix"].nunique(), "unmatched_pct": round(len(unmatched_customer_geo) / len(customers) * 100, 4)},
    {"entity": "sellers", "unmatched_rows": len(unmatched_seller_geo), "unique_unmatched_zip_prefixes": unmatched_seller_geo["seller_zip_code_prefix"].nunique(), "unmatched_pct": round(len(unmatched_seller_geo) / len(sellers) * 100, 4)}
])

display(geo_coverage_review)

,entity,unmatched_rows,unique_unmatched_zip_prefixes,unmatched_pct
0,customers,278,157,0.2796
1,sellers,7,7,0.2262


### Relationship Validation Result

All core business relationships passed without unmatched keys. This includes the links between orders, customers, items, products, sellers, payments, and reviews.

The only relationships marked for review were the joins from customers and sellers to `geolocation_zip_prefix`. The unmatched percentages are very low: 0.2796% for customers and 0.2262% for sellers.

Most unmatched customer rows are concentrated in `DF`, but the overall volume is small. This does not block the relational model. It only means that future geographic analysis should account for a small number of customer or seller zip prefixes without matching latitude and longitude.

The project will continue using `geolocation_zip_prefix_clean.csv` for geographic joins, since it preserves one row per zip prefix and avoids row multiplication.

## Row Multiplication Risks

The previous checks confirm that the main keys and relationships are valid. However, valid relationships do not automatically mean that every join is safe for every type of analysis.

Several tables have a lower grain than `orders`. For example, one order can contain multiple items, multiple payment records, or more than one review row. If these tables are joined directly to `orders`, the number of rows can increase and order-level metrics can become inflated.

This section checks the row impact of the main one-to-many joins. The purpose is to make the risk visible before defining the safe join strategy.

In [10]:
row_multiplication_summary = pd.DataFrame([
    {"join_path": "orders + order_items", "base_rows": len(orders), "rows_after_left_join": len(orders[["order_id"]].merge(order_items[["order_id"]], on="order_id", how="left")), "risk_reason": "One order can contain multiple items."},
    {"join_path": "orders + order_payments", "base_rows": len(orders), "rows_after_left_join": len(orders[["order_id"]].merge(order_payments[["order_id"]], on="order_id", how="left")), "risk_reason": "One order can have multiple payment sequences."},
    {"join_path": "orders + order_reviews", "base_rows": len(orders), "rows_after_left_join": len(orders[["order_id"]].merge(order_reviews[["order_id"]], on="order_id", how="left")), "risk_reason": "One order can have more than one review row."},
    {"join_path": "customers + geolocation_zip_prefix", "base_rows": len(customers), "rows_after_left_join": len(customers[["customer_zip_code_prefix"]].merge(geolocation_zip_prefix[["geolocation_zip_code_prefix"]], left_on="customer_zip_code_prefix", right_on="geolocation_zip_code_prefix", how="left")), "risk_reason": "Safe if using aggregated zip prefix table."},
    {"join_path": "sellers + geolocation_zip_prefix", "base_rows": len(sellers), "rows_after_left_join": len(sellers[["seller_zip_code_prefix"]].merge(geolocation_zip_prefix[["geolocation_zip_code_prefix"]], left_on="seller_zip_code_prefix", right_on="geolocation_zip_code_prefix", how="left")), "risk_reason": "Safe if using aggregated zip prefix table."}
])

row_multiplication_summary["row_increase"] = row_multiplication_summary["rows_after_left_join"] - row_multiplication_summary["base_rows"]
row_multiplication_summary["row_increase_pct"] = (row_multiplication_summary["row_increase"] / row_multiplication_summary["base_rows"] * 100).round(2)
row_multiplication_summary["risk_level"] = pd.cut(row_multiplication_summary["row_increase_pct"], bins=[-0.01, 0, 1, 5, float("inf")], labels=["Low", "Moderate", "Medium", "High"])

display(row_multiplication_summary)

,join_path,base_rows,rows_after_left_join,risk_reason,row_increase,row_increase_pct,risk_level
0,orders + order_items,99441,113425,One order can contain multiple items.,13984,14.06,High
1,orders + order_payments,99441,103887,One order can have multiple payment sequences.,4446,4.47,Medium
2,orders + order_reviews,99441,99992,One order can have more than one review row.,551,0.55,Moderate
3,customers + geolocation_zip_prefix,99441,99441,Safe if using aggregated zip prefix table.,0,0.00,Low
4,sellers + geolocation_zip_prefix,3095,3095,Safe if using aggregated zip prefix table.,0,0.00,Low


### Row Multiplication Result

The row multiplication check confirms that the main risk comes from joining lower-grain tables directly to `orders`.

The largest increase appears when joining `orders` with `order_items`, where the number of rows increases by 14.06%. This is expected because one order can contain multiple items, but it means that item-level data must be aggregated before being used in order-level analysis.

Joining `orders` with `order_payments` also increases the number of rows, although to a smaller extent, with a 4.47% increase. This shows that some orders have multiple payment sequences, so payment data should also be aggregated by `order_id` before order-level joins.

The join between `orders` and `order_reviews` creates only a small row increase of 0.55%, but it still confirms that reviews are not perfectly one-to-one with orders. Review data should therefore be handled carefully, especially because it is post-delivery information and leakage-sensitive for future machine learning work.

The joins with `geolocation_zip_prefix` do not increase row counts. This confirms that the aggregated zip prefix table is the correct geographic reference table for future joins.

## Safe Join Strategy

The previous checks show that the main relationships are valid, but they also confirm that not every valid join is safe for every type of analysis.

For future order-level analysis, `orders` should remain the base table. This is the correct grain for delivery performance analysis, late-delivery calculations, and future machine learning dataset construction.

Tables with a lower grain must be handled carefully:

- `order_items` should be aggregated by `order_id` before joining to order-level datasets.
- `order_payments` should be aggregated by `order_id` before joining to order-level datasets.
- `order_reviews` should only be aggregated for post-delivery analysis and should not be used as a direct feature source for future ML modeling.
- `geolocation_zip_prefix` should be used for customer and seller geographic joins because it preserves one row per zip prefix.
- The raw geolocation table should not be joined directly because it can multiply rows.

For item-level analysis, `order_items` can be used as the base table. This is the appropriate grain for product, seller, price, freight, and category-level questions.

This strategy keeps future analysis aligned with the correct table grain and reduces the risk of inflated metrics caused by accidental row duplication.

## SQL Layer

The relational structure validated in this notebook was also implemented in MySQL Workbench.

The SQL layer uses the cleaned CSV files from `data/processed/` and keeps the database workflow separate from the pandas validation workflow. The scripts are organized for database setup, table creation, import guidance, validation checks, relationship checks, relational views, and reusable business queries.

The SQL implementation confirmed the same relational structure validated in pandas: core business relationships passed, while geolocation joins showed only minor unmatched coverage. Safe order-level aggregation views were also created for items, payments, and reviews.

## Aggregated Order-Level Outputs

The SQL layer already implements safe order-level aggregation views for items, payments, and reviews.

For the pandas workflow, the same logic is also saved as processed CSV outputs. This allows future notebooks to reuse stable one-row-per-order tables without repeating lower-grain joins or depending on a live MySQL connection.

These outputs are relational helper tables, not the final machine learning dataset. Review-related data is included for post-delivery analysis only and remains leakage-sensitive for future modeling.

In [15]:
order_items_agg = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        order_item_count=("order_item_id", "count"),
        product_count=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),
        total_item_price=("price", "sum"),
        total_freight_value=("freight_value", "sum"),
        avg_item_price=("price", "mean"),
        max_item_price=("price", "max")
    )
)

order_items_agg["total_order_item_value"] = order_items_agg["total_item_price"] + order_items_agg["total_freight_value"]

In [16]:
main_payment_type = (
    order_payments
    .groupby(["order_id", "payment_type"])
    .size()
    .reset_index(name="payment_type_count")
    .sort_values(["order_id", "payment_type_count", "payment_type"], ascending=[True, False, True])
    .drop_duplicates("order_id")
    [["order_id", "payment_type"]]
    .rename(columns={"payment_type": "main_payment_type"})
)

payments_agg = (
    order_payments
    .groupby("order_id", as_index=False)
    .agg(
        payment_count=("payment_sequential", "count"),
        payment_method_count=("payment_type", "nunique"),
        total_payment_value=("payment_value", "sum"),
        avg_payment_value=("payment_value", "mean"),
        max_payment_installments=("payment_installments", "max")
    )
    .merge(main_payment_type, on="order_id", how="left")
)

In [17]:
order_reviews_tmp = order_reviews.copy()
order_reviews_tmp["has_review_comment"] = order_reviews_tmp["review_comment_message"].notna().astype(int)

reviews_agg = (
    order_reviews_tmp
    .groupby("order_id", as_index=False)
    .agg(
        review_count=("review_id", "count"),
        avg_review_score=("review_score", "mean"),
        min_review_score=("review_score", "min"),
        max_review_score=("review_score", "max"),
        review_comment_count=("has_review_comment", "sum")
    )
)

In [18]:
aggregated_outputs_summary = pd.DataFrame([
    {"table": "order_items_agg", "rows": len(order_items_agg), "columns": order_items_agg.shape[1], "unique_order_ids": order_items_agg["order_id"].nunique(), "duplicate_order_ids": order_items_agg.duplicated("order_id").sum()},
    {"table": "payments_agg", "rows": len(payments_agg), "columns": payments_agg.shape[1], "unique_order_ids": payments_agg["order_id"].nunique(), "duplicate_order_ids": payments_agg.duplicated("order_id").sum()},
    {"table": "reviews_agg", "rows": len(reviews_agg), "columns": reviews_agg.shape[1], "unique_order_ids": reviews_agg["order_id"].nunique(), "duplicate_order_ids": reviews_agg.duplicated("order_id").sum()}
])

display(aggregated_outputs_summary)

,table,rows,columns,unique_order_ids,duplicate_order_ids
0,order_items_agg,98666,9,98666,0
1,payments_agg,99440,7,99440,0
2,reviews_agg,98673,6,98673,0


## Save Relational Outputs

The aggregated tables are saved to `data/processed/` so they can be reused in later notebooks without repeating lower-grain joins.

Only selected relational outputs are saved. Temporary validation summaries remain inside the notebook unless they are needed as reusable project artifacts.

In [19]:
if aggregated_outputs_summary["duplicate_order_ids"].sum() > 0:
    raise ValueError("At least one aggregated output contains duplicated order_id values.")

In [20]:
order_items_agg[["total_item_price", "total_freight_value", "avg_item_price", "max_item_price", "total_order_item_value"]] = order_items_agg[["total_item_price", "total_freight_value", "avg_item_price", "max_item_price", "total_order_item_value"]].round(2)
payments_agg[["total_payment_value", "avg_payment_value"]] = payments_agg[["total_payment_value", "avg_payment_value"]].round(2)
reviews_agg["avg_review_score"] = reviews_agg["avg_review_score"].round(2)

In [21]:
relational_model_summary = pd.DataFrame([
    {"output_file": "order_items_agg.csv", "source": "order_items", "grain": "One row per order_id", "rows": len(order_items_agg), "purpose": "Safe order-level item aggregation.", "ml_note": "Potentially usable depending on final prediction timing."},
    {"output_file": "payments_agg.csv", "source": "order_payments", "grain": "One row per order_id", "rows": len(payments_agg), "purpose": "Safe order-level payment aggregation.", "ml_note": "Potentially usable after payment approval."},
    {"output_file": "reviews_agg.csv", "source": "order_reviews", "grain": "One row per order_id", "rows": len(reviews_agg), "purpose": "Post-delivery review aggregation.", "ml_note": "Leakage-sensitive. Not suitable as direct ML features."},
    {"output_file": "relationship_validation_summary.csv", "source": "relationship validation", "grain": "One row per checked relationship", "rows": len(relationship_validation_summary), "purpose": "Documents relationship coverage.", "ml_note": "Validation metadata only."},
    {"output_file": "relational_model_summary.csv", "source": "notebook documentation", "grain": "One row per saved relational output", "rows": 5, "purpose": "Documents saved relational outputs.", "ml_note": "Documentation metadata only."}
])

display(relational_model_summary)

,output_file,source,grain,rows,purpose,ml_note
0,order_items_agg.csv,order_items,One row per order_id,98666,Safe order-level item aggregation.,Potentially usable depending on final predicti...
1,payments_agg.csv,order_payments,One row per order_id,99440,Safe order-level payment aggregation.,Potentially usable after payment approval.
2,reviews_agg.csv,order_reviews,One row per order_id,98673,Post-delivery review aggregation.,Leakage-sensitive. Not suitable as direct ML f...
3,relationship_validation_summary.csv,relationship validation,One row per checked relationship,8,Documents relationship coverage.,Validation metadata only.
4,relational_model_summary.csv,notebook documentation,One row per saved relational output,5,Documents saved relational outputs.,Documentation metadata only.


In [22]:
outputs_to_save = {
    "order_items_agg.csv": order_items_agg,
    "payments_agg.csv": payments_agg,
    "reviews_agg.csv": reviews_agg,
    "relationship_validation_summary.csv": relationship_validation_summary,
    "relational_model_summary.csv": relational_model_summary
}

save_summary = []

for file_name, df in outputs_to_save.items():
    output_path = PROCESSED_DIR / file_name
    df.to_csv(output_path, index=False)
    save_summary.append({"file": file_name, "rows": df.shape[0], "columns": df.shape[1], "saved_to": str(output_path.relative_to(PROJECT_ROOT))})

save_summary = pd.DataFrame(save_summary)
display(save_summary)

,file,rows,columns,saved_to
0,order_items_agg.csv,98666,9,data\processed\order_items_agg.csv
1,payments_agg.csv,99440,7,data\processed\payments_agg.csv
2,reviews_agg.csv,98673,6,data\processed\reviews_agg.csv
3,relationship_validation_summary.csv,8,7,data\processed\relationship_validation_summary...
4,relational_model_summary.csv,5,6,data\processed\relational_model_summary.csv


## Notebook Conclusion

This notebook established the relational and SQL-ready foundation for the SupplyGuard project.

The cleaned processed tables were loaded successfully, and the main relational structure was validated through key checks, relationship checks, and row multiplication analysis. Core business relationships passed without unmatched keys, while the only minor coverage limitation appeared in geolocation joins, where a small percentage of customer and seller zip prefixes did not match the aggregated geolocation reference table.

The row multiplication analysis confirmed that `order_items`, `order_payments`, and `order_reviews` should not be joined directly to `orders` for order-level analysis without prior aggregation. Based on this, safe one-row-per-order outputs were created and saved for future notebooks:

- `order_items_agg.csv`
- `payments_agg.csv`
- `reviews_agg.csv`

The SQL layer was also implemented separately in MySQL Workbench, using the same relational logic validated in this notebook. SQL scripts were organized for database setup, table creation, import guidance, quality checks, relationship checks, aggregation views, and reusable business queries.

The project is now ready to move into delivery performance EDA using a reliable relational foundation.